In [2]:
from pathlib import Path
import json, pandas as pd, numpy as np

DATA = Path('./data/')

schema = json.loads((DATA / 'feature_schema_context.json').read_text(encoding='utf-8'))
df = pd.read_parquet(DATA / 'model_features_context.parquet')
print(df.shape)                            # (2629440, 56)

# pd.set_option("display.max_columns", None)
# df
# 
INPUTS = schema['inputs']          # 입력 39개 (직접 고르지 말고 이 목록을 쓰기)
TARGET = 'rentals_target_1h'       # 반납 모델은 'returns_target_1h'

use = df.coverage_status.eq('observed_operating') & df[TARGET].notna()
train = df[use & df.split.eq('train')]
valid = df[use & df.split.eq('validation')]
test  = df[use & df.split.eq('test')]      # 마지막까지 열어보지 않기
print(len(train), len(valid), len(test))

(2629440, 56)
1569378 526178 524875


In [9]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import joblib  # Imported joblib

print("📊 Executing Completely Isolated Seasonal Naive Pipeline...")
INPUTS = schema['inputs']
TARGETS = ['rentals_target_1h', 'returns_target_1h']

# Step 1: Define a Lightweight Seasonal Naive Model Class
class SeasonalNaiveModel:
    def __init__(self):
        # Maps target columns to their respective historical 1-week (168h) lag feature keys
        self.lag_mapping = {
            'rentals_target_1h': 'rentals_lag_168h',
            'returns_target_1h': 'returns_lag_168h'
        }
    
    def predict(self, df_input, target_col):
        """
        Receives a DataFrame and extracts the prediction based on the target column rules.
        """
        if target_col not in self.lag_mapping:
            raise ValueError(f"Unknown target column: {target_col}")
            
        lag_feature = self.lag_mapping[target_col]
        
        # Pull the lag feature, handle missing historical values safely, and clip negative numbers to 0
        predictions = np.clip(df_input[lag_feature].fillna(0.0), 0, None)
        return predictions.values.astype(float)

# Step 2: Instantiate the model
seasonal_naive_model = SeasonalNaiveModel()

# Step 3: Extract clear time-series masks
use = df.coverage_status.eq('observed_operating') & df['rentals_target_1h'].notna() & df['returns_target_1h'].notna()
valid_mask = use & df.split.eq('validation')
test_mask = use & df.split.eq('test')

X_valid_idx = df[valid_mask].index
X_test_idx  = df[test_mask].index

metrics_records = []
pred_df_list = []

# Core evaluation score matrix
def score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    err = y_pred - y_true
    total = np.abs(y_true).sum()
    return {
        'MAE': np.abs(err).mean(), 
        'RMSE': np.sqrt((err**2).mean()),
        'WAPE': np.abs(err).sum() / total if total else np.nan, 
        'Bias': err.mean()
    }

# Explicit segment definitions
segments = {
    '전체': (X_valid_idx, X_test_idx),
    '출근': (X_valid_idx[df.loc[X_valid_idx, 'commute_period'] == 'morning'], X_test_idx[df.loc[X_test_idx, 'commute_period'] == 'morning']),
    '퇴근': (X_valid_idx[df.loc[X_valid_idx, 'commute_period'] == 'evening'], X_test_idx[df.loc[X_test_idx, 'commute_period'] == 'evening']),
    '역세권': (X_valid_idx[df.loc[X_valid_idx, 'station_group'] == '역세권'], X_test_idx[df.loc[X_test_idx, 'station_group'] == '역세권']),
    '업무지구': (X_valid_idx[df.loc[X_valid_idx, 'station_group'] == '업무지구'], X_test_idx[df.loc[X_test_idx, 'station_group'] == '업무지구']),
    '주거지': (X_valid_idx[df.loc[X_valid_idx, 'station_group'] == '주거지'], X_test_idx[df.loc[X_test_idx, 'station_group'] == '주거지'])
}

for target in TARGETS:
    # Use the model object to generate predictions instead of manual series slicing
    valid_predictions = seasonal_naive_model.predict(df.loc[valid_mask], target_col=target)
    test_predictions  = seasonal_naive_model.predict(df.loc[test_mask], target_col=target)
    
    valid_pred_series = pd.Series(valid_predictions, index=X_valid_idx)
    test_pred_series  = pd.Series(test_predictions, index=X_test_idx)
    
    for seg_name, (v_idx, t_idx) in segments.items():
        v_res = score(df.loc[v_idx, target], valid_pred_series.loc[v_idx])
        t_res = score(df.loc[t_idx, target], test_pred_series.loc[t_idx])
        
        for split, res in [('validation', v_res), ('test', t_res)]:
            metrics_records.append({
                'model': 'SeasonalNaive', 'target': target, 'split': split, 'segment': seg_name,
                'n': len(v_idx) if split == 'validation' else len(t_idx),
                'MAE': res['MAE'], 'RMSE': res['RMSE'], 'WAPE': res['WAPE'], 'Bias': res['Bias'], 
                'train_seconds': 0.0
            })

    for split_name, mask, preds in [('validation', valid_mask, valid_pred_series), ('test', test_mask, test_pred_series)]:
        sub_pred = df.loc[mask, ['station_id', 'timestamp']].copy()
        sub_pred['split'] = split_name
        sub_pred['target'] = target
        sub_pred['y_true'] = df.loc[mask, target].astype(float)
        sub_pred['y_pred'] = preds.values.astype(float)
        pred_df_list.append(sub_pred)

# Write output documents and export the pipeline matrix
DATA = Path('./results/')
DATA.mkdir(parents=True, exist_ok=True)

# ---> NEW JOBLIB EXPORT BLOCK <---
joblib.dump(seasonal_naive_model, './models/bicycle_seasonalnaive.pkl')
print(f"💾 Model architecture pickled successfully to {DATA}")

metrics_df = pd.DataFrame(metrics_records)
metrics_df.to_csv(DATA / 'metrics_SeasonalNaive.csv', index=False)

pred_df = pd.concat(pred_df_list, ignore_index=True)
pred_df.to_parquet(DATA / 'pred_SeasonalNaive.parquet', index=False, engine='pyarrow')

with open(DATA / 'params_SeasonalNaive.json', 'w') as f:
    json.dump({"inputs": ['rentals_lag_168h', 'returns_lag_168h'], "hyperparameters": {}}, f)
    
print("🎯 Seasonal Naive Baseline completed with no index dependency or NaN errors!")


📊 Executing Completely Isolated Seasonal Naive Pipeline...
💾 Model architecture pickled successfully to results
🎯 Seasonal Naive Baseline completed with no index dependency or NaN errors!


In [10]:
import pandas as pd

# 1. Load the generated metrics file from your DATA path
metrics_path = DATA / 'metrics_SeasonalNaive.csv'
metrics_df = pd.read_csv(metrics_path)

# 2. Separate Validation and Test for clean viewing
val_summary = metrics_df[metrics_df['split'] == 'validation'].copy()
test_summary = metrics_df[metrics_df['split'] == 'test'].copy()

# 3. Display Validation Dashboard (Use this to compare against Team A's Baseline)
print("=================== 📊 Seasonal Naive 검증 ===================")
val_pivot = val_summary.pivot(index='segment', columns='target', values=['MAE', 'WAPE', 'Bias'])
display(val_pivot.round(4))

# 4. Display Final Test Dashboard (The 2025 Evaluation - Checked Once)
print("\n=================== 🏁 Seasonal Naive 테스트 ===================")
test_pivot = test_summary.pivot(index='segment', columns='target', values=['MAE', 'WAPE', 'Bias'])
display(test_pivot.round(4))


=================== 📊 Seasonal Naive 검증 ===================


MAE                                WAPE  \
target  rentals_target_1h returns_target_1h rentals_target_1h   
segment                                                         
업무지구               2.0391            2.0121            0.7063   
역세권                2.8105            2.7638            0.6459   
전체                 2.1762            2.1387            0.6871   
주거지                1.6787            1.6402            0.7421   
출근                 2.6417            2.7595            0.6228   
퇴근                 3.7645            3.5737            0.6219   

                                       Bias                    
target  returns_target_1h rentals_target_1h returns_target_1h  
segment                                                        
업무지구               0.6634           -0.0017           -0.0021  
역세권                0.6198           -0.0095           -0.0097  
전체                 0.6558           -0.0053           -0.0050  
주거지                0.7160           -0.0047           -0.0031  
출근                 0.5543           -0.0084           -0.0068  
퇴근                 0.5951           -0.0023           -0.0050


=================== 🏁 Seasonal Naive 테스트 ===================


MAE                                WAPE  \
target  rentals_target_1h returns_target_1h rentals_target_1h   
segment                                                         
업무지구               1.7634            1.7326            0.7365   
역세권                2.3814            2.3661            0.6794   
전체                 1.8780            1.8524            0.7217   
주거지                1.4901            1.4595            0.7805   
출근                 2.3357            2.4716            0.6424   
퇴근                 3.2189            3.0006            0.6587   

                                       Bias                    
target  returns_target_1h rentals_target_1h returns_target_1h  
segment                                                        
업무지구               0.6854            0.0030            0.0036  
역세권                0.6535            0.0078            0.0065  
전체                 0.6892            0.0051            0.0047  
주거지                0.7614            0.0046            0.0040  
출근                 0.5670            0.0073            0.0062  
퇴근                 0.6341            0.0066            0.0084